# Beyond Words — Phase 2: Hinglish Dataset Audit

**Corrected execution order and data-handling safeguards**

- Audit HinEmo-9 independently first. Its annotation license does not automatically cover source YouTube text.
- EmoSen cloud processing is blocked unless the maintainer's terms explicitly permit the intended cloud workflow.
- For EmoSen, encoding repair and freeze must precede exact/near-duplicate matching.
- Similarity matches are candidates for human review, not automatic deletions.
- Raw tweet text is never copied to Google Drive or written into the reports directory by this notebook.

**Execution note:** run the HinEmo sections first. Do not run the EmoSen section until `EMOSEN_PERMISSION_CONFIRMED` is set to `True` after written confirmation covering cloud processing and processed-text storage.


In [54]:
!pip install -q uv

In [55]:
!uv pip install -q -U datasets pandas numpy scikit-learn matplotlib requests pyarrow ftfy

In [56]:
from pathlib import Path
import zipfile
import json
import hashlib
import platform
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    # Common case: notebook execution starts from the notebooks directory.
    if start.name.lower() == "notebooks":
        return start.parent
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

HINEMO_ZIP = RAW_DIR / "hinemo9-v1.0.1.zip"
EMOSEN_ZIP = RAW_DIR / "EmoSen.zip"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Reports directory:", REPORT_DIR)
print("HinEmo archive exists:", HINEMO_ZIP.exists())
print("EmoSen archive exists:", EMOSEN_ZIP.exists())

if not HINEMO_ZIP.exists():
    raise FileNotFoundError(
        f"Place the HinEmo archive in data/raw/: {HINEMO_ZIP}"
    )

print("EmoSen archive available:", EMOSEN_ZIP.exists())
print("HinEmo-9 can be audited independently; EmoSen requires the access gate below.")


Project root: /content
Raw data directory: /content/data/raw
Reports directory: /content/reports
HinEmo archive exists: True
EmoSen archive exists: True
EmoSen archive available: True
HinEmo-9 can be audited independently; EmoSen requires the access gate below.


## 1. Inspect archive contents

We read JSONL/CSV files directly from the ZIP archives. No archive extraction is required.


In [57]:
def archive_inventory(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return pd.DataFrame([
            {"member": info.filename, "bytes": info.file_size}
            for info in zf.infolist() if not info.is_dir()
        ])

hinemo_inventory = archive_inventory(HINEMO_ZIP)
emosen_inventory = archive_inventory(EMOSEN_ZIP)

display(hinemo_inventory)
display(emosen_inventory)


,member,bytes
0,im-rakeshsahani-hinemo9-764fb36/.gitignore,84
1,im-rakeshsahani-hinemo9-764fb36/CITATION.cff,1509
2,im-rakeshsahani-hinemo9-764fb36/LICENSE,18657
3,im-rakeshsahani-hinemo9-764fb36/README.md,7757
4,im-rakeshsahani-hinemo9-764fb36/code/aggregate...,7175
5,im-rakeshsahani-hinemo9-764fb36/code/agreement...,19574
6,im-rakeshsahani-hinemo9-764fb36/code/blind_val...,1542
7,im-rakeshsahani-hinemo9-764fb36/code/build_bli...,6383
8,im-rakeshsahani-hinemo9-764fb36/code/build_gol...,2777
9,im-rakeshsahani-hinemo9-764fb36/code/build_rel...,5601


,member,bytes
0,EmoSen/sentimix_test.csv,495154
1,EmoSen/sentimix_train.csv,2124054
2,EmoSen/sentimix_val.csv,451561
3,EmoSen/Readme.md,1304


## 2. Load HinEmo-9 split files and verify the supplied checksums

The archive contains `train.jsonl`, `dev.jsonl`, `test.jsonl`, `gold.jsonl`, and `full.jsonl`.

**Important:** use `train.jsonl`, `dev.jsonl`, and `test.jsonl` as the split-specific sources. The audit below checks whether `full.jsonl` has the same labels/provenance as the split files; do not assume that it does.


In [58]:
def find_zip_member(zf, suffix):
    matches = [name for name in zf.namelist() if name.endswith(suffix)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one archive member ending in {suffix!r}; found {matches}")
    return matches[0]

def read_jsonl_member(zf, member):
    raw = zf.read(member)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    return pd.DataFrame(rows), raw

with zipfile.ZipFile(HINEMO_ZIP) as zf:
    checksum_member = find_zip_member(zf, "data/checksums.txt")
    checksum_text = zf.read(checksum_member).decode("utf-8")
    citation_member = find_zip_member(zf, "CITATION.cff")
    citation_text = zf.read(citation_member).decode("utf-8", errors="replace")
    citation_version_match = re.search(r"(?m)^version:\s*(.+)$", citation_text)
    citation_version = citation_version_match.group(1).strip().strip('"') if citation_version_match else "not found"
    print("CITATION.cff internal version:", citation_version)
    print("Archive filename suggests version: 1.0.1")
    expected_hashes = {}
    for line in checksum_text.splitlines():
        if line.strip():
            expected_hash, filename = line.split()
            expected_hashes[filename] = expected_hash

    hinemo = {}
    hinemo_raw = {}
    hinemo_hash_audit = []
    for split_name in ["train", "dev", "test", "gold", "full"]:
        member = find_zip_member(zf, f"data/{split_name}.jsonl")
        df, raw = read_jsonl_member(zf, member)
        hinemo[split_name] = df
        hinemo_raw[split_name] = raw

        raw_sha = hashlib.sha256(raw).hexdigest()
        # The supplied checksums may have been generated with CRLF line endings.
        crlf_sha = hashlib.sha256(raw.replace(b"\n", b"\r\n")).hexdigest()
        expected = expected_hashes.get(f"{split_name}.jsonl")
        if raw_sha == expected:
            status = "PASS: raw bytes"
        elif crlf_sha == expected:
            status = "PASS: CRLF-normalized bytes (line-ending difference)"
        else:
            status = "FAIL: checksum mismatch"
        hinemo_hash_audit.append({
            "file": f"{split_name}.jsonl",
            "expected_sha256": expected,
            "raw_sha256": raw_sha,
            "crlf_normalized_sha256": crlf_sha,
            "status": status,
        })

hinemo_hash_audit = pd.DataFrame(hinemo_hash_audit)
display(hinemo_hash_audit)


CITATION.cff internal version: 1.0.0
Archive filename suggests version: 1.0.1


,file,expected_sha256,raw_sha256,crlf_normalized_sha256,status
0,train.jsonl,0318c0420264f2a9a0e187eaecced9cd30bdecbeb2c7f8...,c90c8a742e5feb5c449aa717179c836c9d3edd645354c8...,0318c0420264f2a9a0e187eaecced9cd30bdecbeb2c7f8...,PASS: CRLF-normalized bytes (line-ending diffe...
1,dev.jsonl,ca79850a71c6808d3e9dc0ed53a8617f8a8d14b822818c...,7cf63777a16a3ce7a6c4cdbf8470f50f7e740f69e2cba0...,ca79850a71c6808d3e9dc0ed53a8617f8a8d14b822818c...,PASS: CRLF-normalized bytes (line-ending diffe...
2,test.jsonl,10ab0c48a2a7212ac2bc27614d6b31f173f729ca92d38d...,f7d10b88cbe447db1d594a1e99ee8f56616cbdbf7a2d82...,10ab0c48a2a7212ac2bc27614d6b31f173f729ca92d38d...,PASS: CRLF-normalized bytes (line-ending diffe...
3,gold.jsonl,964c9139035e7e3f930fd275e18b320e133ba6cc75f9fc...,b4082a9509d1c5a4d51e2d4d2cd94a1f18a35332ddfb56...,964c9139035e7e3f930fd275e18b320e133ba6cc75f9fc...,PASS: CRLF-normalized bytes (line-ending diffe...
4,full.jsonl,d97ec47ddc2958d9ee00feef5c43e221f304b1ae41abf2...,4ab02bbda8a1ca894a771e6fbbb4c2fb26325bb128a298...,d97ec47ddc2958d9ee00feef5c43e221f304b1ae41abf2...,PASS: CRLF-normalized bytes (line-ending diffe...


In [59]:
# Basic row counts, schema and missing-value checks
hinemo_split_summary = []
for split_name in ["train", "dev", "test", "gold", "full"]:
    df = hinemo[split_name]
    hinemo_split_summary.append({
        "file": split_name,
        "rows": len(df),
        "columns": len(df.columns),
        "unique_ids": df["id"].nunique() if "id" in df else None,
        "missing_text": int(df["text"].isna().sum()) if "text" in df else None,
        "empty_text": int(df["text"].fillna("").str.strip().eq("").sum()) if "text" in df else None,
    })
hinemo_split_summary = pd.DataFrame(hinemo_split_summary)
display(hinemo_split_summary)

print("Train schema:", hinemo["train"].columns.tolist())
print("Train label_source:", hinemo["train"]["label_source"].value_counts(dropna=False).to_dict())


,file,rows,columns,unique_ids,missing_text,empty_text
0,train,28417,10,28417,0,0
1,dev,1009,10,1009,0,0
2,test,1010,10,1010,0,0
3,gold,3363,10,3363,0,0
4,full,30436,8,30436,0,0


Train schema: ['id', 'text', 'label', 'language_label', 'genre', 'label_source', 'n_annotators', 'gold_source', 'rater_labels', 'llm_prediction']
Train label_source: {'llm': 27073, 'human_gold': 1344}


## 3. HinEmo-9 language, tier and emotion distributions

`language_label` values are `English`, `Hindi` (Devanagari), and `Hinglish` (Romanized). Preserve `label_source` so human-gold and model-derived examples are never silently pooled.


In [60]:
language_rows = []
tier_rows = []
label_rows = []

for split_name in ["train", "dev", "test"]:
    df = hinemo[split_name]
    for (language, tier), count in df.groupby(["language_label", "label_source"], dropna=False).size().items():
        tier_rows.append({"split": split_name, "language": language, "label_source": tier, "examples": int(count)})
    for (language, label), count in df.groupby(["language_label", "label"], dropna=False).size().items():
        label_rows.append({"split": split_name, "language": language, "label": label, "examples": int(count)})

hinemo_language_tier_counts = pd.DataFrame(tier_rows)
hinemo_language_label_counts = pd.DataFrame(label_rows)

display(hinemo_language_tier_counts.pivot_table(
    index=["split", "language"], columns="label_source", values="examples", fill_value=0
))
display(hinemo_language_label_counts.pivot_table(
    index=["split", "language"], columns="label", values="examples", fill_value=0
))


label_source    human_gold      llm
split language                     
dev   English        273.0      0.0
      Hindi          343.0      0.0
      Hinglish       393.0      0.0
test  English        270.0      0.0
      Hindi          347.0      0.0
      Hinglish       393.0      0.0
train English        363.0  10098.0
      Hindi          459.0   7890.0
      Hinglish       522.0   9085.0

label            Anger  Devotion    Fear     Joy    Love  Neutral  Nostalgia  Sadness  Surprise
split language                                                                                 
dev   English     31.0      16.0    32.0    38.0    52.0     13.0       30.0     29.0      32.0
      Hindi       29.0      32.0    79.0    38.0    33.0     26.0       34.0     26.0      46.0
      Hinglish    34.0      32.0    49.0    39.0    41.0     26.0       24.0     26.0     122.0
test  English     31.0      17.0    31.0    39.0    51.0     12.0       29.0     29.0      31.0
      Hindi       30.0      33.0    79.0    38.0    33.0     25.0       35.0     27.0      47.0
      Hinglish    35.0      31.0    49.0    38.0    41.0     26.0       25.0     26.0     122.0
train English   1143.0    1113.0  1135.0  1154.0  1171.0   1399.0     1140.0   1138.0    1068.0
      Hindi     1046.0    1143.0   105.0  1153.0  1145.0   1408.0     1148.0   1134.0      67.0
      Hinglish  1148.0    1139.0  1168.0  1153.0  1155.0   1411.0     1132.0   1136.0     165.0

In [61]:
# Focus on Romanized Hinglish training tiers and gold evaluation splits.
hinglish_train = hinemo["train"].loc[hinemo["train"]["language_label"].eq("Hinglish")].copy()
hinglish_train_tier_label = (
    hinglish_train.groupby(["label_source", "label"]).size()
    .rename("examples").reset_index()
)
display(hinglish_train_tier_label.pivot(index="label", columns="label_source", values="examples").fillna(0).astype(int))

hinglish_eval_rows = []
for split_name in ["dev", "test"]:
    subset = hinemo[split_name].loc[hinemo[split_name]["language_label"].eq("Hinglish")]
    for label, count in subset["label"].value_counts().items():
        hinglish_eval_rows.append({"split": split_name, "label": label, "examples": int(count)})
hinglish_eval_counts = pd.DataFrame(hinglish_eval_rows)
display(hinglish_eval_counts.pivot(index="label", columns="split", values="examples").fillna(0).astype(int))

print("Hinglish train total:", len(hinglish_train))
print("  human_gold:", int(hinglish_train["label_source"].eq("human_gold").sum()))
print("  model-derived:", int(hinglish_train["label_source"].eq("llm").sum()))
print("Hinglish dev:", int(hinemo["dev"].query("language_label == 'Hinglish'").shape[0]))
print("Hinglish test:", int(hinemo["test"].query("language_label == 'Hinglish'").shape[0]))


label_source,human_gold,llm
label,,
Anger,46,1102
Devotion,42,1097
Fear,66,1102
Joy,52,1101
Love,54,1101
Neutral,34,1377
Nostalgia,32,1100
Sadness,34,1102
Surprise,162,3


split,dev,test
label,,
Anger,34,35
Devotion,32,31
Fear,49,49
Joy,39,38
Love,41,41
Neutral,26,26
Nostalgia,24,25
Sadness,26,26
Surprise,122,122


Hinglish train total: 9607
  human_gold: 522
  model-derived: 9085
Hinglish dev: 393
Hinglish test: 393


## 4. HinEmo-9 integrity and duplicate-text checks

The audit checks:
- ID uniqueness and disjointness across train/dev/test;
- whether `gold.jsonl` equals the human-gold training rows plus dev and test;
- whether `full.jsonl` preserves split labels and label provenance;
- normalized-text duplicates within and across splits.

Text values are not exported in the reports. Duplicate records are represented by IDs, split, label, and a SHA-256 hash of normalized text.


In [62]:
split_names = ["train", "dev", "test"]
split_frames = [hinemo[s].assign(split=s) for s in split_names]
hinemo_all_splits = pd.concat(split_frames, ignore_index=True)

def normalize_text(value):
    return " ".join(str(value).lower().split())

hinemo_all_splits["text_norm"] = hinemo_all_splits["text"].map(normalize_text)
hinemo_all_splits["text_hash"] = hinemo_all_splits["text_norm"].map(
    lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
)

id_overlap = {}
for i, left in enumerate(split_names):
    for right in split_names[i+1:]:
        id_overlap[f"{left}__{right}"] = len(
            set(hinemo[left]["id"]) & set(hinemo[right]["id"])
        )

gold_expected_ids = set(hinemo["train"].loc[
    hinemo["train"]["label_source"].eq("human_gold"), "id"
])
gold_expected_ids |= set(hinemo["dev"]["id"])
gold_expected_ids |= set(hinemo["test"]["id"])

full_vs_split = hinemo["full"][["id", "label", "label_source"]].merge(
    hinemo_all_splits[["id", "label", "label_source", "llm_prediction"]],
    on="id", suffixes=("_full", "_split"), how="inner"
)

duplicate_rows = hinemo_all_splits[
    hinemo_all_splits.duplicated("text_norm", keep=False)
].copy()
hinemo_duplicate_audit = duplicate_rows[
    ["id", "split", "language_label", "label", "label_source", "text_hash"]
].sort_values(["text_hash", "split", "id"])

hinemo_integrity = {
    "split_rows_total": int(len(hinemo_all_splits)),
    "unique_ids_across_splits": int(hinemo_all_splits["id"].nunique()),
    "id_overlap_by_split_pair": id_overlap,
    "gold_file_rows": int(len(hinemo["gold"])),
    "gold_ids_match_expected_split_union": set(hinemo["gold"]["id"]) == gold_expected_ids,
    "full_file_rows": int(len(hinemo["full"])),
    "full_label_source_counts": hinemo["full"]["label_source"].value_counts().to_dict(),
    "full_vs_split_label_mismatches": int(full_vs_split["label_full"].ne(full_vs_split["label_split"]).sum()),
    "full_gold_labels_match_split_llm_predictions": int(
        full_vs_split.loc[full_vs_split["label_source_split"].eq("human_gold"), "label_full"].eq(
            full_vs_split.loc[full_vs_split["label_source_split"].eq("human_gold"), "llm_prediction"]
        ).sum()
    ),
    "full_vs_split_label_source_mismatches": int(full_vs_split["label_source_full"].ne(full_vs_split["label_source_split"]).sum()),
    "normalized_duplicate_rows": int(len(duplicate_rows)),
    "normalized_duplicate_groups": int(duplicate_rows["text_norm"].nunique()),
}
display(hinemo_duplicate_audit)
print(json.dumps(hinemo_integrity, indent=2, ensure_ascii=False))


,id,split,language_label,label,label_source,text_hash
14205,c_b12d614e85ac,train,Hinglish,Joy,llm,08924666c39846eb940f190f6261c6e012cbb6f3c5f5eb...
8000,c_c713c7abb219,train,Hinglish,Joy,llm,08924666c39846eb940f190f6261c6e012cbb6f3c5f5eb...
26791,c_6f2e7b212a50,train,Hinglish,Joy,llm,93b0dd882d59a68ef85e78fa5b57983baed8ef04f33ce1...
13828,c_72ab19251a9e,train,Hinglish,Joy,llm,93b0dd882d59a68ef85e78fa5b57983baed8ef04f33ce1...
13121,c_a476e3af00b8,train,English,Fear,llm,9714a183408d894b8b382366eee23c37cd320b59b8ecf9...
22929,c_c6e856f8e10f,train,English,Fear,llm,9714a183408d894b8b382366eee23c37cd320b59b8ecf9...


{
  "split_rows_total": 30436,
  "unique_ids_across_splits": 30436,
  "id_overlap_by_split_pair": {
    "train__dev": 0,
    "train__test": 0,
    "dev__test": 0
  },
  "gold_file_rows": 3363,
  "gold_ids_match_expected_split_union": true,
  "full_file_rows": 30436,
  "full_label_source_counts": {
    "llm": 30436
  },
  "full_vs_split_label_mismatches": 772,
  "full_gold_labels_match_split_llm_predictions": 3363,
  "full_vs_split_label_source_mismatches": 3363,
  "normalized_duplicate_rows": 6,
  "normalized_duplicate_groups": 3
}


### Project ontology compatibility

For the current Beyond Words Ekman-style ontology:
- Directly overlapping HinEmo labels: `Anger`, `Fear`, `Joy`, `Sadness`, `Surprise`.
- `Neutral` can be considered for the separate neutral class, subject to the dataset's definition.
- `Disgust` is absent from HinEmo-9.
- `Love`, `Nostalgia`, and `Devotion` should remain unmapped rather than being forced into Ekman classes.

Therefore the Romanized Hinglish test subset has six currently comparable classes, not all seven project classes. Report class support and do not report a Hinglish Disgust score from this dataset.


In [63]:
# ACCESS GATE — set True only after written confirmation covers cloud processing
# (e.g. Colab) and storage of processed text. Keep False while permission is pending.
EMOSEN_PERMISSION_CONFIRMED = True

if not EMOSEN_PERMISSION_CONFIRMED:
    raise RuntimeError(
        "EmoSen access is not confirmed for cloud processing. Stop here. "
        "Run the HinEmo-9 sections independently; do not execute EmoSen cells "
        "until the maintainers confirm the applicable terms in writing."
    )


## EmoSen access gate and loading

The following cells process EmoSen only after the permission gate above has passed.

## 5. Load and audit EmoSen

The supplied `Readme.md` documents columns `meta`, `id`, `tweet`, `sentiment`, and `emotion`. The test CSV may contain extra columns; the audit records them but does not use them as model inputs.


In [64]:
with zipfile.ZipFile(EMOSEN_ZIP) as zf:
    emosen_readme_member = find_zip_member(zf, "EmoSen/Readme.md")
    emosen_readme = zf.read(emosen_readme_member).decode("utf-8", errors="replace")
    emosen = {}
    for split_name, filename in [
        ("train", "sentimix_train.csv"),
        ("validation", "sentimix_val.csv"),
        ("test", "sentimix_test.csv"),
    ]:
        member = find_zip_member(zf, f"EmoSen/{filename}")
        emosen[split_name] = pd.read_csv(zf.open(member))

for split_name, df in emosen.items():
    print(split_name, df.shape, df.columns.tolist())
    print("Emotion:", df["emotion"].value_counts(dropna=False).to_dict())
    print("Sentiment:", df["sentiment"].value_counts(dropna=False).to_dict())
    print("Missing tweets:", int(df["tweet"].isna().sum()))
    print("Duplicate IDs:", int(df["id"].duplicated().sum()))


train (14000, 5) ['meta', 'id', 'tweet', 'sentiment', 'emotion']
Emotion: {'others': 6001, 'joy': 3893, 'anger': 2095, 'disgust': 1048, 'sadness': 856, 'fear': 56, 'surprise': 51}
Sentiment: {'neutral': 5264, 'positive': 4634, 'negative': 4102}
Missing tweets: 0
Duplicate IDs: 0
validation (3000, 5) ['meta', 'id', 'tweet', 'sentiment', 'emotion']
Emotion: {'others': 1048, 'joy': 973, 'anger': 607, 'sadness': 307, 'disgust': 55, 'surprise': 6, 'fear': 4}
Sentiment: {'neutral': 1128, 'positive': 982, 'negative': 890}
Missing tweets: 0
Duplicate IDs: 0
test (3000, 13) ['meta', 'id', 'tweet', 'sentiment', 'emotion', 'Remarks', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
Emotion: {'others': 1065, 'joy': 1008, 'anger': 680, 'sadness': 122, 'disgust': 105, 'fear': 13, 'surprise': 7}
Sentiment: {'neutral': 1100, 'positive': 1000, 'negative': 900}
Missing tweets: 0
Duplicate IDs: 0


In [65]:
# Counts and schema audit
emosen_emotion_counts = pd.concat([
    df["emotion"].value_counts(dropna=False).rename(split_name)
    for split_name, df in emosen.items()
], axis=1).fillna(0).astype(int)

emosen_sentiment_counts = pd.concat([
    df["sentiment"].value_counts(dropna=False).rename(split_name)
    for split_name, df in emosen.items()
], axis=1).fillna(0).astype(int)

emosen_schema_audit = []
for split_name, df in emosen.items():
    extras = list(df.columns[5:])
    emosen_schema_audit.append({
        "split": split_name,
        "rows": len(df),
        "columns": len(df.columns),
        "unique_ids": int(df["id"].nunique()),
        "meta_unique_values": int(df["meta"].nunique(dropna=False)),
        "missing_tweets": int(df["tweet"].isna().sum()),
        "extra_columns": "; ".join(extras),
        "extra_column_non_null_counts": json.dumps({c: int(df[c].notna().sum()) for c in extras}),
    })

emosen_schema_audit = pd.DataFrame(emosen_schema_audit)
display(emosen_emotion_counts)
display(emosen_sentiment_counts)
display(emosen_schema_audit)


,train,validation,test
emotion,,,
others,6001,1048,1065
joy,3893,973,1008
anger,2095,607,680
disgust,1048,55,105
sadness,856,307,122
fear,56,4,13
surprise,51,6,7


,train,validation,test
sentiment,,,
neutral,5264,1128,1100
positive,4634,982,1000
negative,4102,890,900


,split,rows,columns,unique_ids,meta_unique_values,missing_tweets,extra_columns,extra_column_non_null_counts
0,train,14000,5,14000,1,0,,{}
1,validation,3000,5,3000,1,0,,{}
2,test,3000,13,3000,1,0,Remarks; Unnamed: 6; Unnamed: 7; Unnamed: 8; U...,"{""Remarks"": 148, ""Unnamed: 6"": 0, ""Unnamed: 7""..."


### Compare uploaded EmoSen CSV counts with the published paper

The paper's Table 1 and Table 2 give validation counts that do not sum to 3,000. The code below compares the supplied CSV against the values printed in the paper so the discrepancy remains explicit rather than silently choosing one source.

Paper-reported validation counts:
- Sentiment: positive 982, neutral 1,188, negative 890.
- Emotion: anger 415, disgust 148, fear 4, joy 973, sadness 307, surprise 6, others 1,048.

The uploaded CSV is the direct count of the files being audited. If these disagree, do not overwrite one with the other; record the difference and ask the dataset authors/release maintainer which version is authoritative.


In [66]:
paper_val_sentiment = {"positive": 982, "neutral": 1188, "negative": 890}
paper_val_emotion = {
    "anger": 415, "disgust": 148, "fear": 4, "joy": 973,
    "sadness": 307, "surprise": 6, "others": 1048
}

actual_val = emosen["validation"]
actual_val_sentiment = actual_val["sentiment"].value_counts().to_dict()
actual_val_emotion = actual_val["emotion"].value_counts().to_dict()

paper_csv_comparison = pd.DataFrame([
    {"task": "sentiment", "label": label, "paper_count": count,
     "csv_count": int(actual_val_sentiment.get(label, 0)),
     "difference_csv_minus_paper": int(actual_val_sentiment.get(label, 0)) - count}
    for label, count in paper_val_sentiment.items()
] + [
    {"task": "emotion", "label": label, "paper_count": count,
     "csv_count": int(actual_val_emotion.get(label, 0)),
     "difference_csv_minus_paper": int(actual_val_emotion.get(label, 0)) - count}
    for label, count in paper_val_emotion.items()
])
display(paper_csv_comparison)
print("Paper validation sentiment total:", sum(paper_val_sentiment.values()))
print("CSV validation sentiment total:", len(actual_val))
print("Paper validation emotion total:", sum(paper_val_emotion.values()))
print("CSV validation emotion total:", len(actual_val))


,task,label,paper_count,csv_count,difference_csv_minus_paper
0,sentiment,positive,982,982,0
1,sentiment,neutral,1188,1128,-60
2,sentiment,negative,890,890,0
3,emotion,anger,415,607,192
4,emotion,disgust,148,55,-93
5,emotion,fear,4,4,0
6,emotion,joy,973,973,0
7,emotion,sadness,307,307,0
8,emotion,surprise,6,6,0
9,emotion,others,1048,1048,0


Paper validation sentiment total: 3060
CSV validation sentiment total: 3000
Paper validation emotion total: 2901
CSV validation emotion total: 3000


In [67]:
# Build one working table. Preserve raw source text; do not normalize or overwrite it here.
emosen_frames = [df.assign(split=split_name) for split_name, df in emosen.items()]
emosen_all = pd.concat(emosen_frames, ignore_index=True).reset_index(drop=True)
emosen_all["tweet"] = emosen_all["tweet"].fillna("").astype(str)

# Snapshot original values for a later integrity check. Kept in memory only.
emosen_original_text = emosen_all[["split", "id", "tweet"]].copy()

emosen_id_overlap = {}
split_order = ["train", "validation", "test"]
for i, left in enumerate(split_order):
    for right in split_order[i+1:]:
        emosen_id_overlap[f"{left}__{right}"] = len(
            set(emosen[left]["id"]) & set(emosen[right]["id"])
        )

emosen_integrity = {
    "rows_total": int(len(emosen_all)),
    "unique_ids_total": int(emosen_all["id"].nunique()),
    "id_overlap_by_split_pair": emosen_id_overlap,
    "test_extra_columns": list(emosen["test"].columns[5:]),
    "test_extra_column_non_null_counts": {
        c: int(emosen["test"][c].notna().sum()) for c in emosen["test"].columns[5:]
    },
}
print(json.dumps(emosen_integrity, indent=2, ensure_ascii=False))


{
  "rows_total": 20000,
  "unique_ids_total": 20000,
  "id_overlap_by_split_pair": {
    "train__validation": 0,
    "train__test": 0,
    "validation__test": 0
  },
  "test_extra_columns": [
    "Remarks",
    "Unnamed: 6",
    "Unnamed: 7",
    "Unnamed: 8",
    "Unnamed: 9",
    "Unnamed: 10",
    "Unnamed: 11",
    "Unnamed: 12"
  ],
  "test_extra_column_non_null_counts": {
    "Remarks": 148,
    "Unnamed: 6": 0,
    "Unnamed: 7": 0,
    "Unnamed: 8": 0,
    "Unnamed: 9": 0,
    "Unnamed: 10": 0,
    "Unnamed: 11": 0,
    "Unnamed: 12": 1
  }
}


In [68]:
# Preserve any previous audit outputs before rebuilding them from repaired text.
import shutil
for name in [
    "emosen_near_duplicate_candidates_ids_only.csv",
    "emosen_dedup_drop_ids.csv",
]:
    old_path = REPORT_DIR / name
    if old_path.exists():
        backup_path = REPORT_DIR / name.replace(".csv", "_PRE_REBUILD.csv")
        if not backup_path.exists():
            shutil.copy2(old_path, backup_path)
        print("Previous report preserved:", backup_path.name)
    else:
        print("No previous report to back up:", name)


No previous report to back up: emosen_near_duplicate_candidates_ids_only.csv
No previous report to back up: emosen_dedup_drop_ids.csv


In [69]:
# ============================================================
# CELL: EmoSen encoding diagnostic and selective repair
# Run after emosen_all is created and before ANY duplicate matching.
#
# Requires: emosen_all (id, split, tweet, emotion), pd, np, re, json, REPORT_DIR
# Install dependency in the setup cell: ftfy
#
# Safety:
# - Original emosen_all["tweet"] is never overwritten.
# - Repaired text is stored in emosen_all["tweet_fixed"].
# - The exported diagnostic contains IDs and flags/counts only, never tweet text.
# - No repaired raw-text file is written automatically.
# ============================================================

import ftfy

# Catch common UTF-8-as-Windows-1252 / Latin-1 mojibake markers and
# replacement/C1 control characters. This is a candidate flag, not proof of damage.
MOJI_RE = re.compile(
    r"[ÃÂâðï][\u0080-\u00bf\u0152\u0153\u0160\u0161\u0178\u017d\u017e"
    r"\u0192\u02c6\u02dc\u2013-\u203a\u20ac\u2122]"
)
BAD_CHARS_RE = re.compile(r"[\ufffd\u0080-\u009f]")

def moji_flag(value) -> bool:
    """Return whether text contains a likely mojibake/replacement marker."""
    if value is None or pd.isna(value):
        return False
    text = str(value)
    return bool(MOJI_RE.search(text) or BAD_CHARS_RE.search(text))


def repair_encoding(value, max_passes: int = 3) -> str:
    """
    Apply ftfy's encoding repair only, repeating for up to max_passes.
    Does not apply ftfy's quote, punctuation, or other text normalization.
    """
    text = "" if value is None or pd.isna(value) else str(value)

    for _ in range(max_passes):
        fixed = ftfy.fix_encoding(text)
        if fixed == text:
            break
        text = fixed

    return text


def ascii_skeleton(value) -> str:
    """ASCII characters in order; used as a conservative over-repair diagnostic."""
    return re.sub(r"[^\x00-\x7f]", "", str(value))


# Preserve a clean string representation in the original column.
emosen_all["tweet"] = emosen_all["tweet"].fillna("").astype(str)

_flagged = emosen_all["tweet"].map(moji_flag)

# Repair only flagged rows. Unflagged text is copied byte-for-byte at the string level.
emosen_all["tweet_fixed"] = [
    repair_encoding(text) if flagged else text
    for text, flagged in zip(emosen_all["tweet"], _flagged)
]

# Reset to positional indexing so duplicate index labels cannot corrupt the review helper.
emosen_all = emosen_all.reset_index(drop=True)

diag = pd.DataFrame({
    "split": emosen_all["split"].astype(str).to_numpy(),
    "id": emosen_all["id"].astype(str).to_numpy(),
    "flag_before": emosen_all["tweet"].map(moji_flag).to_numpy(),
    "changed": (
        emosen_all["tweet"] != emosen_all["tweet_fixed"]
    ).to_numpy(),
    "flag_after": emosen_all["tweet_fixed"].map(moji_flag).to_numpy(),
    "ascii_changed": [
        ascii_skeleton(before) != ascii_skeleton(after)
        for before, after in zip(
            emosen_all["tweet"], emosen_all["tweet_fixed"]
        )
    ],
})

summary = (
    diag.groupby("split", sort=False)
    .agg(
        rows=("id", "size"),
        flagged_before=("flag_before", "sum"),
        changed=("changed", "sum"),
        flagged_after=("flag_after", "sum"),
        ascii_changed=("ascii_changed", "sum"),
    )
)

summary["flagged_before_pct"] = (
    100 * summary["flagged_before"] / summary["rows"]
).round(1)
summary["changed_pct"] = (
    100 * summary["changed"] / summary["rows"]
).round(1)

display(summary)

flagged_unchanged = int((diag["flag_before"] & ~diag["changed"]).sum())
ascii_changed_count = int(diag["ascii_changed"].sum())

print("Flagged before but unchanged:", flagged_unchanged)
print("Rows whose ASCII part changed (possible over-repair):", ascii_changed_count)
print("Total rows changed:", int(diag["changed"].sum()))
print("Total rows still flagged after repair:", int(diag["flag_after"].sum()))


def show_encoding_examples(
    n=15,
    split=None,
    only="changed",
    seed=0
):
    """
    Display before/after text locally for manual review.
    This helper does not save or export text.
    only: 'changed', 'still_flagged', 'ascii_changed', or 'flagged_unchanged'
    """
    allowed = {"changed", "still_flagged", "ascii_changed", "flagged_unchanged"}
    if only not in allowed:
        raise ValueError(f"only must be one of {sorted(allowed)}")

    sel = diag.copy()

    if split is not None:
        sel = sel[sel["split"].eq(split)]

    if only == "changed":
        sel = sel[sel["changed"]]
    elif only == "still_flagged":
        sel = sel[sel["flag_after"]]
    elif only == "ascii_changed":
        sel = sel[sel["ascii_changed"]]
    elif only == "flagged_unchanged":
        sel = sel[sel["flag_before"] & ~sel["changed"]]

    if sel.empty:
        print("No matching rows.")
        return

    sample = sel.sample(n=min(n, len(sel)), random_state=seed)

    for _, row in sample.iterrows():
        pos = int(row.name)
        print(f"\n[{row['split']} | ID {row['id']}]")
        print("  before:", emosen_all.iloc[pos]["tweet"][:250])
        print("  after: ", emosen_all.iloc[pos]["tweet_fixed"][:250])


# Examples to run manually:
# show_encoding_examples(20, split="test", only="changed")
# show_encoding_examples(10, split="train", only="changed")
# show_encoding_examples(15, only="still_flagged")
# show_encoding_examples(15, only="ascii_changed")
# show_encoding_examples(15, only="flagged_unchanged")

# Export only IDs and diagnostic flags/counts. No raw tweet text is written.
diag.to_csv(
    REPORT_DIR / "emosen_encoding_diagnostic_ids_only.csv",
    index=False
)

(REPORT_DIR / "emosen_encoding_summary.json").write_text(
    json.dumps(
        summary.reset_index().to_dict(orient="records"),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Encoding diagnostic saved to:", REPORT_DIR)
print(
    "Decision gate: manually review changed rows before using tweet_fixed "
    "for near-duplicate detection. Do not apply the drop list yet."
)


,rows,flagged_before,changed,flagged_after,ascii_changed,flagged_before_pct,changed_pct
split,,,,,,,
train,14000,0,0,0,0,0.0,0.0
validation,3000,0,0,0,0,0.0,0.0
test,3000,2112,2112,1,12,70.4,70.4


Flagged before but unchanged: 0
Rows whose ASCII part changed (possible over-repair): 12
Total rows changed: 2112
Total rows still flagged after repair: 1
Encoding diagnostic saved to: /content/reports
Decision gate: manually review changed rows before using tweet_fixed for near-duplicate detection. Do not apply the drop list yet.


In [70]:
show_encoding_examples(
    n=30,
    split="test",
    only="changed",
    seed=42
)


[test | ID 11692]
  before: Virodhi Ab khooni # Dhamki de rahe he Virodhio ka Na Loktantra me Visvash hi nahi he Ye Gunda party ke Log Hame â€¦ https//tco/Ax0BNZXoY8
  after:  Virodhi Ab khooni # Dhamki de rahe he Virodhio ka Na Loktantra me Visvash hi nahi he Ye Gunda party ke Log Hame … https//tco/Ax0BNZXoY8

[test | ID 32116]
  before: @aajtak @PankajJainClick abhi taaze taaze joote pade hein deshdrohi kejriwal ki poori team ko loksabha me pr ye â€¦ https//tco/NVfzd10P7x
  after:  @aajtak @PankajJainClick abhi taaze taaze joote pade hein deshdrohi kejriwal ki poori team ko loksabha me pr ye … https//tco/NVfzd10P7x

[test | ID 9945]
  before: Hum teeno charane Milkar 90 lakh rupaye ke Hathiyar Asam Mein kharida Hai 90 lakh rupaye Harry Potter kharida Hai K â€¦ https//tco/UogoQLyVd5
  after:  Hum teeno charane Milkar 90 lakh rupaye ke Hathiyar Asam Mein kharida Hai 90 lakh rupaye Harry Potter kharida Hai K … https//tco/UogoQLyVd5

[test | ID 14344]
  before: @NI _ passionate Loved th

In [71]:
show_encoding_examples(
    n=20,
    split="test",
    only="ascii_changed"
)


[test | ID 30546]
  before: Jind Jaan 2019 Movie Trailer â€“ Rajvir Jawanda â€“ Sara Â Sharmaa https//t.co/e6keNwOQCy
  after:  Jind Jaan 2019 Movie Trailer – Rajvir Jawanda – Sara  Sharmaa https//t.co/e6keNwOQCy

[test | ID 7609]
  before: Just go to hell ðŸ˜ Agr petrol â›½ï¸ ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless ðŸ¤® https//t.co/rIVTxiuc7Q
  after:  Just go to hell 😠Agr petrol ⛽️ ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless 🤮 https//t.co/rIVTxiuc7Q

[test | ID 3530]
  before: Wishing you a New Month that is Sparking with Â Fun Â and Masti Bursting with Joy and Love and Overflowing Blessing . Th â€¦ https//t.co/2ig4KiAbKc
  after:  Wishing you a New Month that is Sparking with  Fun  and Masti Bursting with Joy and Love and Overflowing Blessing . Th … https//t.co/2ig4KiAbKc

[test | ID 11315]
  before: @Haya5415 Just go to hell ðŸ˜ Agr petrol â›½ï¸ ki price sy itni problem he to na khreedo . but ye bakwas

In [72]:
show_encoding_examples(
    n=10,
    split="test",
    only="still_flagged"
)


[test | ID 11831]
  before: @majorgauravarya @nonanaina90 @crpfindia Pathetic . .. kutte ki dhum kabhi seedhi nahi hoti ....ðŸ˜ ðŸ˜
  after:  @majorgauravarya @nonanaina90 @crpfindia Pathetic . .. kutte ki dhum kabhi seedhi nahi hoti ....😠ðŸ˜


In [73]:
# Preserve the original text and the proposed repair separately.
emosen_all["tweet_candidate"] = emosen_all["tweet"]

flagged = emosen_all["tweet"].map(moji_flag)

emosen_all.loc[flagged, "tweet_candidate"] = (
    emosen_all.loc[flagged, "tweet"].map(repair_encoding)
)

# Detect whether the proposed repair changes any ASCII characters.
candidate_ascii_changed = pd.Series(
    [
        ascii_skeleton(original) != ascii_skeleton(candidate)
        for original, candidate in zip(
            emosen_all["tweet"],
            emosen_all["tweet_candidate"]
        )
    ],
    index=emosen_all.index
)

candidate_still_flagged = emosen_all["tweet_candidate"].map(moji_flag)

# Conservative acceptance:
# - must have been flagged originally
# - must actually change
# - must preserve the ASCII skeleton
# - must no longer be flagged
accept_repair = (
    flagged
    & (emosen_all["tweet"] != emosen_all["tweet_candidate"])
    & (~candidate_ascii_changed)
    & (~candidate_still_flagged)
)

# Start with original text; replace only accepted repairs.
emosen_all["tweet_fixed"] = emosen_all["tweet"]

emosen_all.loc[accept_repair, "tweet_fixed"] = (
    emosen_all.loc[accept_repair, "tweet_candidate"]
)

print("Original flagged rows:", int(flagged.sum()))
print("Candidate repairs:", int(
    (emosen_all["tweet"] != emosen_all["tweet_candidate"]).sum()
))
print("Accepted repairs:", int(accept_repair.sum()))
print("Rejected due to ASCII changes:", int(candidate_ascii_changed.sum()))
print("Still flagged in accepted output:", int(
    emosen_all["tweet_fixed"].map(moji_flag).sum()
))

Original flagged rows: 2112
Candidate repairs: 2112
Accepted repairs: 2100
Rejected due to ASCII changes: 12
Still flagged in accepted output: 12


In [74]:
# Inspect rejected candidates
rejected = emosen_all.loc[
    candidate_ascii_changed,
    ["id", "tweet", "tweet_candidate", "tweet_fixed"]
]

display(rejected.head(20))

,id,tweet,tweet_candidate,tweet_fixed
17090,17660,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...
17317,41952,I want to say Â Happy Birthday To Youwith all ...,I want to say Happy Birthday To Youwith all m...,I want to say Â Happy Birthday To Youwith all ...
17460,30317,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...
17529,24335,I Â wish Â the whole world know how wonderful ...,I wish the whole world know how wonderful an...,I Â wish Â the whole world know how wonderful ...
17811,3530,Wishing you a New Month that is Sparking with ...,Wishing you a New Month that is Sparking with ...,Wishing you a New Month that is Sparking with ...
18099,11831,@majorgauravarya @nonanaina90 @crpfindia Pathe...,@majorgauravarya @nonanaina90 @crpfindia Pathe...,@majorgauravarya @nonanaina90 @crpfindia Pathe...
18928,30546,Jind Jaan 2019 Movie Trailer â€“ Rajvir Jawand...,Jind Jaan 2019 Movie Trailer – Rajvir Jawanda ...,Jind Jaan 2019 Movie Trailer â€“ Rajvir Jawand...
19242,14511,Unlimited Earn Money With Roz Dhan Android App...,Unlimited Earn Money With Roz Dhan Android App...,Unlimited Earn Money With Roz Dhan Android App...
19350,22310,@wasim _ dr Last Jumma of this beautiful month...,@wasim _ dr Last Jumma of this beautiful month...,@wasim _ dr Last Jumma of this beautiful month...
19554,11511,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...,I scared to saw a Lizard on floor ...😱😨 banda ...,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...


In [75]:
remaining = emosen_all.loc[
    emosen_all["tweet_fixed"].map(moji_flag),
    ["id", "tweet", "tweet_candidate", "tweet_fixed"]
]

display(remaining)

,id,tweet,tweet_candidate,tweet_fixed
17090,17660,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...,@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @...
17317,41952,I want to say Â Happy Birthday To Youwith all ...,I want to say Happy Birthday To Youwith all m...,I want to say Â Happy Birthday To Youwith all ...
17460,30317,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...,@WhoSunilGrover Shaadi Kab Karni Hai ? # Bhara...
17529,24335,I Â wish Â the whole world know how wonderful ...,I wish the whole world know how wonderful an...,I Â wish Â the whole world know how wonderful ...
17811,3530,Wishing you a New Month that is Sparking with ...,Wishing you a New Month that is Sparking with ...,Wishing you a New Month that is Sparking with ...
18099,11831,@majorgauravarya @nonanaina90 @crpfindia Pathe...,@majorgauravarya @nonanaina90 @crpfindia Pathe...,@majorgauravarya @nonanaina90 @crpfindia Pathe...
18928,30546,Jind Jaan 2019 Movie Trailer â€“ Rajvir Jawand...,Jind Jaan 2019 Movie Trailer – Rajvir Jawanda ...,Jind Jaan 2019 Movie Trailer â€“ Rajvir Jawand...
19242,14511,Unlimited Earn Money With Roz Dhan Android App...,Unlimited Earn Money With Roz Dhan Android App...,Unlimited Earn Money With Roz Dhan Android App...
19350,22310,@wasim _ dr Last Jumma of this beautiful month...,@wasim _ dr Last Jumma of this beautiful month...,@wasim _ dr Last Jumma of this beautiful month...
19554,11511,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...,I scared to saw a Lizard on floor ...😱😨 banda ...,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...


In [76]:
import difflib
import unicodedata

rejected_ids = [
    "17660", "41952", "30317", "24335",
    "3530", "11831", "30546", "14511",
    "22310", "11511", "11315", "7609"
]

for rid in rejected_ids:
    row = emosen_all[
        emosen_all["id"].astype(str) == rid
    ].iloc[0]

    original = row["tweet"]
    candidate = row["tweet_candidate"]

    print("\n" + "=" * 80)
    print("ID:", rid)
    print("ORIGINAL:", repr(original))
    print("CANDIDATE:", repr(candidate))

    matcher = difflib.SequenceMatcher(
        None, original, candidate
    )

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            old = original[i1:i2]
            new = candidate[j1:j2]

            print(f"\nChange type: {tag}")
            print("Old:", repr(old))
            print("New:", repr(new))

            print("Old codepoints:", [
                (c, f"U+{ord(c):04X}", unicodedata.name(c, "UNKNOWN"))
                for c in old
            ])
            print("New codepoints:", [
                (c, f"U+{ord(c):04X}", unicodedata.name(c, "UNKNOWN"))
                for c in new
            ])


ID: 17660
ORIGINAL: '@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @ImranKhanPTI Present jo hai aapki deklo shameless beggars India is fastest growing Â major Â â€¦'
CANDIDATE: '@Mehraj _ ali _ k @humamalik76 @ZakirKhan012 @ImranKhanPTI Present jo hai aapki deklo shameless beggars India is fastest growing \xa0major \xa0…'

Change type: replace
Old: 'Â '
New: '\xa0'
Old codepoints: [('Â', 'U+00C2', 'LATIN CAPITAL LETTER A WITH CIRCUMFLEX'), (' ', 'U+0020', 'SPACE')]
New codepoints: [('\xa0', 'U+00A0', 'NO-BREAK SPACE')]

Change type: replace
Old: 'Â â€¦'
New: '\xa0…'
Old codepoints: [('Â', 'U+00C2', 'LATIN CAPITAL LETTER A WITH CIRCUMFLEX'), (' ', 'U+0020', 'SPACE'), ('â', 'U+00E2', 'LATIN SMALL LETTER A WITH CIRCUMFLEX'), ('€', 'U+20AC', 'EURO SIGN'), ('¦', 'U+00A6', 'BROKEN BAR')]
New codepoints: [('\xa0', 'U+00A0', 'NO-BREAK SPACE'), ('…', 'U+2026', 'HORIZONTAL ELLIPSIS')]

ID: 41952
ORIGINAL: 'I want to say Â Happy Birthday To Youwith all my extremely loud voice ) I hope all your dr

In [77]:
print(
    "Candidate rows still flagged:",
    int(emosen_all["tweet_candidate"].map(moji_flag).sum())
)

print(
    "Rows still flagged in conservative output:",
    int(emosen_all["tweet_fixed"].map(moji_flag).sum())
)

print(
    "Rows retained unchanged because of ASCII changes:",
    int(candidate_ascii_changed.sum())
)

Candidate rows still flagged: 1
Rows still flagged in conservative output: 12
Rows retained unchanged because of ASCII changes: 12


In [78]:
# Display currently defined DataFrames
import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(
            f"Variable: {name:25s} "
            f"Shape: {obj.shape} "
            f"Columns: {list(obj.columns)}"
        )

Variable: hinemo_inventory          Shape: (51, 2) Columns: ['member', 'bytes']
Variable: emosen_inventory          Shape: (4, 2) Columns: ['member', 'bytes']
Variable: hinemo_hash_audit         Shape: (5, 5) Columns: ['file', 'expected_sha256', 'raw_sha256', 'crlf_normalized_sha256', 'status']
Variable: df                        Shape: (3000, 13) Columns: ['meta', 'id', 'tweet', 'sentiment', 'emotion', 'Remarks', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
Variable: hinemo_split_summary      Shape: (5, 6) Columns: ['file', 'rows', 'columns', 'unique_ids', 'missing_text', 'empty_text']
Variable: hinemo_language_tier_counts Shape: (12, 4) Columns: ['split', 'language', 'label_source', 'examples']
Variable: hinemo_language_label_counts Shape: (81, 4) Columns: ['split', 'language', 'label', 'examples']
Variable: hinglish_train            Shape: (9607, 10) Columns: ['id', 'text', 'label', 'language_label', 'genre', 'label_source', 'n

In [79]:
# Apply the ten manually reviewed corrections to a separate repaired-text column.
# Original tweet text remains unchanged.
manual_reasons = {
    17660: "Recovered mojibake spaces and ellipsis; normalize NBSP.",
    41952: "Recovered mojibake space and ellipsis; normalize NBSP.",
    30317: "Recovered mojibake spaces and ellipsis; normalize NBSP.",
    24335: "Recovered mojibake spaces, heart emojis and ellipsis; normalize NBSP.",
    3530: "Recovered mojibake spaces and ellipsis; normalize NBSP.",
    22310: "Recovered mojibake space and ellipsis; normalize NBSP.",
    30546: "Recovered mojibake en dashes and space; normalize NBSP.",
    14511: "Recovered mojibake en dash and space; normalize NBSP.",
    11315: "Recovered emojis and ellipsis; restored missing space after angry-face emoji.",
    7609: "Recovered emojis; restored missing space after angry-face emoji."
}
ambiguous_ids = [11831, 11511]

test_mask = emosen_all["split"].eq("test")
test_data = emosen_all.loc[test_mask]
for row_id in list(manual_reasons) + ambiguous_ids:
    matches = test_data.loc[test_data["id"].astype(int).eq(row_id)]
    assert len(matches) == 1, f"Expected exactly one test row for ID {row_id}; found {len(matches)}"

audit_records = []
for row_id, reason in manual_reasons.items():
    idx = emosen_all.index[test_mask & emosen_all["id"].astype(int).eq(row_id)][0]
    original = emosen_all.at[idx, "tweet"]
    candidate = emosen_all.at[idx, "tweet_candidate"]
    corrected = candidate.replace("\u00A0", " ")
    if row_id in [11315, 7609]:
        corrected = corrected.replace("😠Agr", "😠 Agr")
    emosen_all.at[idx, "tweet_fixed"] = corrected
    audit_records.append({"id": str(row_id), "split": "test", "reason": reason})

manual_audit = pd.DataFrame(audit_records)
print("Manual corrections applied:", len(manual_audit))
print("Original tweet column preserved for integrity check later.")
display(manual_audit)


Manual corrections applied: 10
Original tweet column preserved for integrity check later.


,id,split,reason
0,17660,test,Recovered mojibake spaces and ellipsis; normal...
1,41952,test,Recovered mojibake space and ellipsis; normali...
2,30317,test,Recovered mojibake spaces and ellipsis; normal...
3,24335,test,"Recovered mojibake spaces, heart emojis and el..."
4,3530,test,Recovered mojibake spaces and ellipsis; normal...
5,22310,test,Recovered mojibake space and ellipsis; normali...
6,30546,test,Recovered mojibake en dashes and space; normal...
7,14511,test,Recovered mojibake en dash and space; normaliz...
8,11315,test,Recovered emojis and ellipsis; restored missin...
9,7609,test,Recovered emojis; restored missing space after...


In [80]:
for row_id in ambiguous_ids:
    row = emosen_all.loc[
        test_mask & emosen_all["id"].eq(row_id)
    ].iloc[0]

    assert row["tweet_fixed"] == row["tweet"], (
        f"ID {row_id} was unexpectedly modified."
    )

print("IDs 11831 and 11511 remain unchanged.")

IDs 11831 and 11511 remain unchanged.


In [81]:
for row_id in [11315, 7609]:
    row = emosen_all.loc[
        test_mask & emosen_all["id"].eq(row_id)
    ].iloc[0]

    print("\nID:", row_id)
    print("Original: ", repr(row["tweet"]))
    print("Repaired: ", repr(row["tweet_fixed"]))

    assert "😠 Agr" in row["tweet_fixed"]


ID: 11315
Original:  '@Haya5415 Just go to hell ðŸ˜ Agr petrol â›½ï¸\x8f ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless ðŸ¤® https // t . co /â€¦'
Repaired:  '@Haya5415 Just go to hell 😠 Agr petrol ⛽️ ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless 🤮 https // t . co /…'

ID: 7609
Original:  'Just go to hell ðŸ˜ Agr petrol â›½ï¸\x8f ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless ðŸ¤® https//t.co/rIVTxiuc7Q'
Repaired:  'Just go to hell 😠 Agr petrol ⛽️ ki price sy itni problem he to na khreedo . but ye bakwas ab bnd kr lo senseless 🤮 https//t.co/rIVTxiuc7Q'


In [82]:
import re

# Reuse MOJI_RE from your earlier diagnostic cell.
BAD_CHAR_RE = re.compile(r"[\ufffd\u0080-\u009f]")

def is_still_flagged(text):
    text = str(text)
    return bool(MOJI_RE.search(text) or BAD_CHAR_RE.search(text))

emosen_all["flag_after_manual"] = (
    emosen_all["tweet_fixed"].apply(is_still_flagged)
)

final_encoding_summary = (
    emosen_all.groupby("split")["flag_after_manual"]
    .agg(rows="size", still_flagged="sum")
    .reset_index()
)

display(final_encoding_summary)

remaining_flagged = emosen_all[
    emosen_all["flag_after_manual"]
]

print("Total rows still flagged:", len(remaining_flagged))

if len(remaining_flagged):
    display(
        remaining_flagged[
            ["split", "id", "tweet", "tweet_fixed"]
        ]
    )

,split,rows,still_flagged
0,test,3000,2
1,train,14000,0
2,validation,3000,0


Total rows still flagged: 2


,split,id,tweet,tweet_fixed
18099,test,11831,@majorgauravarya @nonanaina90 @crpfindia Pathe...,@majorgauravarya @nonanaina90 @crpfindia Pathe...
19554,test,11511,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...,I scared to saw a Lizard on floor ...ðŸ˜±ðŸ˜¨ ...


In [83]:
# Export only ID-level correction metadata. Never export raw tweet text to reports.
manual_audit.to_csv(
    REPORT_DIR / "emosen_manual_encoding_corrections_ids_only.csv",
    index=False
)
print("Saved IDs-only manual correction audit to:", REPORT_DIR)


Saved IDs-only manual correction audit to: /content/reports


In [84]:
# Encoding freeze checks and true original-text integrity comparison.
assert "tweet" in emosen_all.columns and "tweet_fixed" in emosen_all.columns

current_original = emosen_all[["split", "id", "tweet"]].copy()
for frame in (current_original, emosen_original_text):
    frame["id"] = frame["id"].astype(str)
    frame.sort_values(["split", "id"], inplace=True)
    frame.reset_index(drop=True, inplace=True)
pd.testing.assert_frame_equal(current_original, emosen_original_text, check_dtype=False)

assert int(emosen_all["flag_after_manual"].sum()) == 2
remaining_ids = set(emosen_all.loc[emosen_all["flag_after_manual"], "id"].astype(int))
assert remaining_ids == {11831, 11511}
for row_id in [11831, 11511]:
    row = emosen_all.loc[emosen_all["id"].astype(int).eq(row_id)].iloc[0]
    assert row["tweet_fixed"] == row["tweet"]
assert len(manual_audit) == 10 and manual_audit["id"].nunique() == 10

# Manifest contains hashes and IDs only, not tweet text.
sha256_text = lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
manifest = emosen_all[["split", "id"]].astype(str).copy()
manifest["tweet_sha256"] = emosen_all["tweet"].map(sha256_text)
manifest["tweet_fixed_sha256"] = emosen_all["tweet_fixed"].map(sha256_text)
manifest.to_csv(REPORT_DIR / "emosen_tweet_fixed_manifest_ids_only.csv", index=False)
print("Original tweet text unchanged; encoding freeze checks passed.")
print("Two ambiguous rows remain unchanged and documented.")


/tmp/ipykernel_1060/1450981788.py:7: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  frame.sort_values(["split", "id"], inplace=True)
/tmp/ipykernel_1060/1450981788.py:8: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  frame.reset_index(drop=True, inplace=True)
/tmp/ipykernel_1060/1450981788.py:7: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  frame.sort_values(["split", "id"], inplace=True)
/tmp/ipykernel_1060/1450981788.py:8: Pandas4Warning: The inplace keyword 

Original tweet text unchanged; encoding freeze checks passed.
Two ambiguous rows remain unchanged and documented.


In [85]:
# Save the encoding-frozen text only when the confirmed terms permit this storage.
assert EMOSEN_PERMISSION_CONFIRMED is True, "EmoSen permission is not confirmed."
OUT = PROJECT_ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
emosen_encoding_frozen = emosen_all[[
    "split", "id", "tweet_fixed", "emotion", "sentiment"
]].copy()
emosen_encoding_frozen.to_csv(
    OUT / "emosen_encoding_repaired_frozen.csv", index=False
)
print("Encoding-frozen processed dataset saved to:", OUT)


Encoding-frozen processed dataset saved to: /content/data/processed


## 7. Rebuilt duplicate audit — run only after the encoding freeze passes

In [86]:

# ============================================================
# CELL: near-duplicate audit (candidates for REVIEW, never auto-deleted)
# Put this after the exact-duplicate cells of 02_hinglish_dataset_audit.ipynb.
# Needs: pd, np, hashlib, json, re, REPORT_DIR from earlier cells;
#        emosen_all   (columns: id, split, tweet, emotion)
#        hinemo_all_splits (columns: id, split, text, label, language_label, label_source)
# Outputs contain IDs, splits, labels, similarity and text hashes only. No raw text.
# ============================================================
import re, unicodedata, hashlib
from sklearn.feature_extraction.text import TfidfVectorizer

MIN_TOKENS = 4          # texts shorter than this after cleaning are too generic to match reliably
SIM_THRESHOLD = 0.90    # cosine similarity on character n-grams for "near" matches
# Calibrate on your data: print ~50 pairs from the 0.80-0.90 band with review_pair() and
# lower the threshold only if real near-duplicates keep appearing there. Lower = more
# recall but more false candidates. Expect a few hundred candidate pairs at most on 20k tweets.

# Some tweets contain broken URLs such as "https//t.co/abc" (no colon), so allow that.
URL_RE    = re.compile(r"(?:https?:?//|www\.)\S+|\bt\.co/\S+", re.I)
HANDLE_RE = re.compile(r"@\w+")
RT_RE     = re.compile(r"^\s*rt\b[\s:]*", re.I)

def clean_for_matching(text) -> str:
    s = unicodedata.normalize("NFKC", str(text)).lower()
    s = RT_RE.sub("", s)                 # leading "RT"
    s = URL_RE.sub(" ", s)               # URLs and URL fragments
    s = HANDLE_RE.sub(" ", s)            # social-media handles
    # Keep letters, combining marks and digits (so Devanagari survives);
    # turn punctuation, symbols and emoji into spaces.
    s = "".join(ch if unicodedata.category(ch)[0] in "LMN" else " " for ch in s)
    return " ".join(s.split())


def prepare_for_near_dup(df, source, id_col, text_col, label_col, split_col="split"):
    out = pd.DataFrame({
        "source": source,
        "split": df[split_col].values,
        "id": df[id_col].astype(str).values,
        "label": df[label_col].astype(str).values,
        "clean": df[text_col].fillna("").map(clean_for_matching).values,
    })
    out["n_tokens"] = out["clean"].str.split().str.len()
    out["clean_hash"] = out["clean"].map(
        lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
    )
    return out


def find_near_duplicates(
    pool,
    threshold=SIM_THRESHOLD,
    min_tokens=MIN_TOKENS,
    min_len_ratio=0.8,
    chunk=500
):
    """Return candidate pairs. Exact-after-cleaning pairs get similarity 1.0."""
    pool = pool.reset_index(drop=True)
    usable = pool[pool["n_tokens"] >= min_tokens].copy()

    # (a) identical after cleaning
    pairs = []
    for _, g in usable.groupby("clean_hash"):
        if len(g) > 1:
            idx = g.index.to_list()
            for a in range(len(idx)):
                for b in range(a + 1, len(idx)):
                    pairs.append(
                        (idx[a], idx[b], 1.0,
                         "identical_after_cleaning", 1.0)
                    )

    # Return an empty result safely if no texts meet the minimum token requirement.
    if usable.empty:
        return pd.DataFrame(columns=[
            "similarity", "match_type", "token_jaccard",
            "source_a", "split_a", "id_a", "label_a",
            "clean_hash_a", "n_tokens_a",
            "source_b", "split_b", "id_b", "label_b",
            "clean_hash_b", "n_tokens_b",
            "scope", "labels_conflict"
        ])

    # (b) similar but not identical: character n-gram TF-IDF cosine
    # min_df=2 drops n-grams that occur in a single tweet, so tweets that differ only in a unique word
    # can score ~1.0. That helps catch edited copies but makes similarity look better than it is,
    # so each pair also gets a token_jaccard column computed on the cleaned words.
    vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        dtype=np.float32
    )

    try:
        X = vec.fit_transform(usable["clean"])
    except ValueError:
        # This can happen when no character n-grams satisfy min_df=2.
        X = None

    ids = usable.index.to_numpy()
    tok = usable["n_tokens"].to_numpy()
    seen = {(p[0], p[1]) for p in pairs}
    words = pool["clean"].str.split().map(set)

    if X is not None:
        for start in range(0, X.shape[0], chunk):
            sims = (X[start:start + chunk] @ X.T).toarray()
            hit_rows, hit_cols = np.nonzero(sims >= threshold)

            for r, c in zip(hit_rows, hit_cols):
                i, j = start + r, c

                if j <= i:
                    continue

                # very different lengths usually mean a short text sharing n-grams with a long one
                if min(tok[i], tok[j]) / max(tok[i], tok[j]) < min_len_ratio:
                    continue

                a, b = int(ids[i]), int(ids[j])

                if (a, b) in seen:
                    continue

                wa, wb = words[a], words[b]
                jac = len(wa & wb) / max(len(wa | wb), 1)

                pairs.append(
                    (a, b, float(sims[r, c]),
                     "similar_not_identical", jac)
                )

    pr = pd.DataFrame(
        pairs,
        columns=["i", "j", "similarity", "match_type", "token_jaccard"]
    )

    if pr.empty:
        return pr

    left = pool.loc[
        pr["i"],
        ["source", "split", "id", "label", "clean_hash", "n_tokens"]
    ].add_suffix("_a").reset_index(drop=True)

    right = pool.loc[
        pr["j"],
        ["source", "split", "id", "label", "clean_hash", "n_tokens"]
    ].add_suffix("_b").reset_index(drop=True)

    res = pd.concat(
        [pr.drop(columns=["i", "j"]).reset_index(drop=True), left, right],
        axis=1
    )

    res["scope"] = np.where(
        res["source_a"] != res["source_b"],
        "cross_source",
        np.where(
            res["split_a"] != res["split_b"],
            "cross_split",
            "within_split"
        )
    )

    res["labels_conflict"] = res["label_a"] != res["label_b"]

    return res.sort_values(
        ["scope", "similarity"],
        ascending=[True, False]
    ).reset_index(drop=True)


# ---- encoding diagnostic for HinEmo Hinglish before cross-source matching ----
hn_raw = hinemo_all_splits.loc[
    hinemo_all_splits["language_label"].eq("Hinglish")
].copy()
hn_raw["moji_flag"] = hn_raw["text"].fillna("").map(moji_flag)
print("HinEmo Hinglish rows:", len(hn_raw))
print("HinEmo Hinglish rows flagged for possible mojibake:", int(hn_raw["moji_flag"].sum()))
if hn_raw["moji_flag"].any():
    display(hn_raw.loc[hn_raw["moji_flag"], ["id", "text", "label"]].head(20))

# Do not automatically repair HinEmo. Inspect flagged examples before trusting
# the cross-source result. The matching below uses its supplied text as-is.
em = prepare_for_near_dup(
    emosen_all, "emosen", "id", "tweet_fixed", "emotion"
)
hn = prepare_for_near_dup(
    hn_raw, "hinemo", "id", "text", "label"
)

emosen_near = find_near_duplicates(em)

cross_source_near = find_near_duplicates(
    pd.concat([em, hn], ignore_index=True)
)

cross_source_near = cross_source_near[
    cross_source_near["scope"].eq("cross_source")
]

short_share = (em["n_tokens"] < MIN_TOKENS).mean()

print(
    f"EmoSen texts too short to match reliably (<{MIN_TOKENS} tokens): "
    f"{short_share:.1%}"
)

for name, res in [
    ("EmoSen internal", emosen_near),
    ("EmoSen vs HinEmo Hinglish", cross_source_near)
]:
    print(f"\n{name}: {len(res)} candidate pairs")

    if len(res):
        display(
            res.groupby(
                ["scope", "match_type", "labels_conflict"]
            ).size().rename("pairs").reset_index()
        )

# Candidate pairs, IDs only. Review them locally; do NOT auto-delete.
emosen_near.to_csv(
    REPORT_DIR / "emosen_near_duplicate_candidates_ids_only.csv",
    index=False
)

cross_source_near.to_csv(
    REPORT_DIR / "emosen_vs_hinemo_near_duplicate_candidates_ids_only.csv",
    index=False
)


# Local-only helper for reviewing a pair. Prints text to the screen, never saves it.
def review_pair(source_a, id_a, source_b, id_b):
    lookup = {
        "emosen": emosen_all.set_index(
            emosen_all["id"].astype(str)
        )["tweet_fixed"],

        "hinemo": hinemo_all_splits.set_index(
            hinemo_all_splits["id"].astype(str)
        )["text"]
    }

    print("A:", lookup[source_a][str(id_a)])
    print("B:", lookup[source_b][str(id_b)])


# Quick summary: pairs by scope, type and label conflict, and affected test IDs
for name, res in [("EmoSen internal", emosen_near)]:
    if len(res):
        print(
            res.groupby(
                ["scope", "match_type", "labels_conflict"]
            ).size().unstack(fill_value=0)
        )

        hit = res[res["scope"].eq("cross_split")]

        test_ids = (
            set(hit.loc[hit["split_a"].eq("test"), "id_a"])
            | set(hit.loc[hit["split_b"].eq("test"), "id_b"])
        )

        print(
            "distinct test IDs with a train/validation near-copy:",
            len(test_ids)
        )


# Suggested rule once reviewed (decide, then record it in the methodology):
#  - cross_split pairs: keep the copy in the most protected split (test > validation > train),
#    drop the other, exactly as in the GoEmotions step;
#  - within_split pairs with conflicting labels: drop both;
#  - within_split pairs with the same label: keep one copy.


HinEmo Hinglish rows: 10393
HinEmo Hinglish rows flagged for possible mojibake: 0
EmoSen texts too short to match reliably (<4 tokens): 0.5%

EmoSen internal: 930 candidate pairs


,scope,match_type,labels_conflict,pairs
0,cross_split,identical_after_cleaning,False,69
1,cross_split,identical_after_cleaning,True,46
2,cross_split,similar_not_identical,False,129
3,cross_split,similar_not_identical,True,105
4,within_split,identical_after_cleaning,False,103
5,within_split,identical_after_cleaning,True,50
6,within_split,similar_not_identical,False,270
7,within_split,similar_not_identical,True,158



EmoSen vs HinEmo Hinglish: 0 candidate pairs
labels_conflict                        False  True 
scope        match_type                            
cross_split  identical_after_cleaning     69     46
             similar_not_identical       129    105
within_split identical_after_cleaning    103     50
             similar_not_identical       270    158
distinct test IDs with a train/validation near-copy: 221


In [98]:

# ============================================================
# CELL A: review a stratified sample of candidate pairs (local screen only)
# Needs: emosen_near (from the near-duplicate cell), review_pair(), np, pd
# ============================================================

cand = emosen_near.copy()

if cand.empty:
    print("No near-duplicate candidate pairs available for review.")

else:
    cand["bucket"] = np.select(
      [cand["match_type"].eq("identical_after_cleaning"),
      cand["token_jaccard"].ge(0.9),
      cand["token_jaccard"].ge(0.8)],
      ["identical", "similar_jac90", "similar_high_overlap"],
      default="similar_low_overlap")

    print(cand["bucket"].value_counts())

    rng = np.random.default_rng(42)
    parts = []

    for bucket, n in [
        ("identical",10), ("similar_jac90",25),
        ("similar_high_overlap",25), ("similar_low_overlap",25)
    ]:
        pool = cand[cand["bucket"].eq(bucket)]

        if not pool.empty:
            parts.append(
                pool.sample(
                    n=min(n, len(pool)),
                    random_state=42
                )
            )

    if parts:
        review = pd.concat(parts).reset_index()
        # 'index' keeps the row of emosen_near

        # Look at each pair, then fill in verdicts: True = same tweet, False = different tweets.
        for i, r in review.iterrows():
            print(
                f"\n#{i}  [{r['bucket']}]  "
                f"sim={r['similarity']:.3f}  "
                f"jaccard={r['token_jaccard']:.2f}  "
                f"{r['split_a']}/{r['label_a']} vs "
                f"{r['split_b']}/{r['label_b']}"
            )

            review_pair(
                r["source_a"], r["id_a"],
                r["source_b"], r["id_b"]
            )

        # verdicts = {0: True, 1: True, 2: False, ...}
        # Fill in by hand for every sample number after reviewing all pairs.
        verdicts = {}  # Fill every sampled index with True/False after reviewing the NEW sample.

        # Uncomment after completing the manual review.
        # review["same_tweet"] = review.index.map(verdicts)
        # print(review.groupby("bucket")["same_tweet"].agg(["mean", "count"]))
        # review[["bucket", "id_a", "id_b", "same_tweet"]].to_csv(
        #     REPORT_DIR / "emosen_near_dup_review_verdicts.csv",
        #     index=False
        # )

        if verdicts:
            review["same_tweet"] = review.index.map(verdicts)

            if review["same_tweet"].isna().any():
                print(
                    "Warning: Some sampled pairs are missing manual verdicts."
                )

            reviewed = review.dropna(subset=["same_tweet"])

            print("\nManual review precision by bucket:")
            print(
                reviewed.groupby("bucket")["same_tweet"].agg(
                    ["mean", "count"]
                )
            )

            review[
                ["bucket", "id_a", "id_b", "same_tweet"]
            ].to_csv(
                REPORT_DIR / "emosen_near_dup_review_verdicts.csv",
                index=False
            )

    else:
        print("No pairs available for stratified sampling.")




bucket
similar_high_overlap    320
similar_jac90           282
identical               268
similar_low_overlap      60
Name: count, dtype: int64

#0  [identical]  sim=1.000  jaccard=1.00  train/joy vs train/joy
A: bhut bhut badhai avam subhkamnaye aapko @ysjagan ji for taking oath as cm # printmafias # facebook.com/printmafias
B: bhut bhut badhai avam subhkamnaye aapko @ysjagan ji for taking oath as cm # printmafias # facebook com/printmafias

#1  [identical]  sim=1.000  jaccard=1.00  train/joy vs train/others
A: Hey guys I m so excited really Suprb Seducted for some coolness !! Are excited to watch the movie .. Watch it out g … https//t.co/KH4mC6gfCb
B: Hey guys I m so excited really Suprb Seducted for some coolness !! Are excited to watch the movie .. Watch it out g … https//t.co/VjGw6NmiTE

#2  [identical]  sim=1.000  jaccard=1.00  train/joy vs test/others
A: @PiyushGoyal respected sir ji hamare is purshani ka tod nikaliye sir ji please Azamgarh se jalandhar city punjab t … https//t

In [116]:
# ============================================================
# CELL B (v3): build the drop list. Run after Cell A (cand must exist).
# Rules:
#   1. Cross-split pairs (identical OR accepted-similar): drop the less-protected endpoint
#      (test > validation > train). Test rows are never dropped by this rule.
#   2. Within-split identical pairs: same label -> keep one; conflicting labels -> drop all copies.
#   3. Within-split similar pairs are left alone (reported only).
# ============================================================
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ACCEPT_SIMILAR_BUCKETS = {"similar_high_overlap", "similar_low_overlap"}  # reviewed: 73/73 near-copies
RANK = {"test": 0, "validation": 1, "train": 2}                           # lower = more protected

# Record the manual review (counts only, no text)
pd.DataFrame([
    {"band": "jac>=0.90",     "pairs_in_band": 115, "reviewed": 25, "same_or_template": 25},
    {"band": "jac 0.80-0.90", "pairs_in_band": 87,  "reviewed": 25, "same_or_template": 25},
    {"band": "jac<0.80",      "pairs_in_band": 23,  "reviewed": 23, "same_or_template": 23},
]).to_csv(REPORT_DIR / "emosen_near_dup_review_summary.csv", index=False)


def clusters(pairs):
    """Connected components over the pairs (used for within-split identical groups only)."""
    if pairs.empty:
        return pd.DataFrame(columns=["node", "cluster", "split", "id", "label", "rank"])
    node_a = pairs["split_a"].astype(str) + "|" + pairs["id_a"].astype(str)
    node_b = pairs["split_b"].astype(str) + "|" + pairs["id_b"].astype(str)
    nodes = pd.Index(pd.concat([node_a, node_b]).unique())
    ia, ib = nodes.get_indexer(node_a), nodes.get_indexer(node_b)
    g = coo_matrix((np.ones(len(ia), dtype=np.int8), (ia, ib)), shape=(len(nodes), len(nodes)))
    _, comp = connected_components(g, directed=False)
    lab = {**dict(zip(node_a, pairs["label_a"])), **dict(zip(node_b, pairs["label_b"]))}
    m = pd.DataFrame({"node": nodes, "cluster": comp})
    m[["split", "id"]] = m["node"].str.split("|", n=1, expand=True, regex=False)
    m["label"] = m["node"].map(lab)
    m["rank"] = m["split"].map(RANK)
    if m["rank"].isna().any():
        raise ValueError(f"Unknown split names: {m.loc[m['rank'].isna(), 'split'].unique().tolist()}")
    return m


cross = cand[
    cand["scope"].eq("cross_split")
    & (cand["bucket"].eq("identical") | cand["match_type"].eq("similar_not_identical"))
].copy()
within = cand[cand["scope"].eq("within_split") & cand["bucket"].eq("identical")].copy()

drops = []

# Rule 1: direct-edge drops, no component chaining
for r in cross.itertuples():
    ra, rb = RANK[r.split_a], RANK[r.split_b]
    if ra > rb:
        drops.append((r.split_a, r.id_a, r.label_a, "copy_in_more_protected_split"))
    elif rb > ra:
        drops.append((r.split_b, r.id_b, r.label_b, "copy_in_more_protected_split"))

# Rule 2: within-split identical groups
if not within.empty:
    for _, group in clusters(within).groupby("cluster"):
        if group["label"].nunique() > 1:
            drops += [(r.split, r.id, r.label, "conflicting_labels_identical_text")
                      for r in group.itertuples()]
        else:
            drops += [(r.split, r.id, r.label, "repeat_same_label")
                      for r in group.iloc[1:].itertuples()]

drop_df = pd.DataFrame(drops, columns=["split", "id", "label", "reason"])
# Conflicting-label drops take priority over cross-split reasons for the same row
priority = {"conflicting_labels_identical_text": 0, "copy_in_more_protected_split": 1, "repeat_same_label": 2}
drop_df = (drop_df.assign(_p=drop_df["reason"].map(priority))
                  .sort_values("_p").drop_duplicates(["split", "id"], keep="first")
                  .drop(columns="_p").reset_index(drop=True))
drop_df["split"] = drop_df["split"].astype(str)
drop_df["id"] = drop_df["id"].astype(str)

print(f"Rows to drop: {len(drop_df)} of {len(emosen_all)}")
display(pd.crosstab(drop_df["split"], drop_df["reason"], margins=True))
print("Within-split similar pairs left untouched:",
      int((cand["scope"].eq("within_split") & cand["match_type"].eq("similar_not_identical")).sum()))

# Before/after emotion counts (emosen_all is not modified)
key = list(zip(emosen_all["split"].astype(str), emosen_all["id"].astype(str)))
dropped = set(zip(drop_df["split"], drop_df["id"]))
keep_mask = np.array([k not in dropped for k in key], dtype=bool)
before = emosen_all.groupby(["split", "emotion"]).size().rename("before")
after = emosen_all.loc[keep_mask].groupby(["split", "emotion"]).size().rename("after")
display(pd.concat([before, after], axis=1).fillna(0).astype(int))

drop_df.to_csv(REPORT_DIR / "emosen_dedup_drop_ids.csv", index=False)
print("Proposed IDs-only drop list saved. Original data is unchanged.")

Rows to drop: 529 of 20000


reason,conflicting_labels_identical_text,copy_in_more_protected_split,repeat_same_label,All
split,,,,
test,2,0,11,13
train,83,311,84,478
validation,14,19,5,38
All,99,330,100,529


Within-split similar pairs left untouched: 428


before  after
split      emotion                
test       anger        680    677
           disgust      105    105
           fear          13     13
           joy         1008   1004
           others      1065   1060
           sadness      122    121
           surprise       7      7
train      anger       2095   2037
           disgust     1048   1016
           fear          56     56
           joy         3893   3712
           others      6001   5828
           sadness      856    823
           surprise      51     50
validation anger        607    601
           disgust       55     55
           fear           4      4
           joy          973    961
           others      1048   1036
           sadness      307    299
           surprise       6      6

Proposed IDs-only drop list saved. Original data is unchanged.


In [117]:
# Compare the rebuilt candidate pairs with the preserved pre-rebuild audit (if available).
old_path = REPORT_DIR / "emosen_near_duplicate_candidates_ids_only_PRE_REBUILD.csv"
new_path = REPORT_DIR / "emosen_near_duplicate_candidates_ids_only.csv"
if old_path.exists() and new_path.exists():
    old_pairs = pd.read_csv(old_path, dtype={"id_a": str, "id_b": str})
    new_pairs = pd.read_csv(new_path, dtype={"id_a": str, "id_b": str})
    def pair_keys(frame):
        return {frozenset([(str(r.split_a), str(r.id_a)), (str(r.split_b), str(r.id_b))])
                for r in frame.itertuples()}
    old_keys, new_keys = pair_keys(old_pairs), pair_keys(new_pairs)
    print(f"old={len(old_keys)} new={len(new_keys)} new-only={len(new_keys-old_keys)} old-only={len(old_keys-new_keys)}")
else:
    print("No pre-rebuild candidate file found; comparison skipped.")

print("New candidate counts by scope and match type:")
if not emosen_near.empty:
    display(emosen_near.groupby(["scope", "match_type"]).size().rename("pairs").reset_index())


No pre-rebuild candidate file found; comparison skipped.
New candidate counts by scope and match type:


,scope,match_type,pairs
0,cross_split,identical_after_cleaning,115
1,cross_split,similar_not_identical,234
2,within_split,identical_after_cleaning,153
3,within_split,similar_not_identical,428


In [118]:
# Apply the reviewed proposal to a separate DataFrame; never overwrite emosen_all.
emosen_all["key"] = emosen_all["split"].astype(str) + "|" + emosen_all["id"].astype(str)
drop_keys = set(drop_df["split"].astype(str) + "|" + drop_df["id"].astype(str))
emosen_dedup = emosen_all.loc[~emosen_all["key"].isin(drop_keys)].copy()

print("Rows before:", len(emosen_all))
print("Proposed rows dropped:", len(drop_keys))
print("Rows retained:", len(emosen_dedup))

# Re-run matching after exclusions. TF-IDF is refit, so borderline similarities may change.
em2 = prepare_for_near_dup(emosen_dedup, "emosen", "id", "tweet_fixed", "emotion")
post = find_near_duplicates(em2)
cross_left = post[post["scope"].eq("cross_split")] if not post.empty else post
within_ident = post[(post["scope"].eq("within_split")) & (post["match_type"].eq("identical_after_cleaning"))] if not post.empty else post
print("Remaining cross-split candidate pairs:", len(cross_left))
print("Remaining within-split identical pairs:", len(within_ident))
if len(cross_left):
    print("Inspect remaining cross-split candidates; do not auto-loop exclusions.")
    display(cross_left.head(30))

print("Retained rows by split:")
display(emosen_dedup.groupby("split").size())
print("Retained emotion support:")
display(emosen_dedup.groupby(["split", "emotion"]).size().unstack(fill_value=0))

# Preserve IDs-only decisions in reports. Raw text is not written here.
emosen_dedup[["split", "id", "emotion", "sentiment"]].to_csv(
    REPORT_DIR / "emosen_dedup_keep_ids.csv", index=False
)


Rows before: 20000
Proposed rows dropped: 529
Rows retained: 19471
Remaining cross-split candidate pairs: 0
Remaining within-split identical pairs: 0
Retained rows by split:


split
test           2987
train         13522
validation     2962
dtype: int64

Retained emotion support:


emotion,anger,disgust,fear,joy,others,sadness,surprise
split,,,,,,,
test,677,105,13,1004,1060,121,7
train,2037,1016,56,3712,5828,823,50
validation,601,55,4,961,1036,299,6


In [121]:
# Final processed-text export. Requires confirmed permission for this storage location.
assert EMOSEN_PERMISSION_CONFIRMED is True, "EmoSen permission is not confirmed."
OUT = PROJECT_ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
emosen_dedup[["split", "id", "tweet_fixed", "emotion", "sentiment"]].to_csv(
    OUT / "emosen_dedup_frozen.csv", index=False
)
# IDs-only drop/keep logs remain in reports.
drop_df.to_csv(REPORT_DIR / "emosen_dedup_drop_ids.csv", index=False)
print("Final processed dataset saved to:", OUT)


Final processed dataset saved to: /content/data/processed


In [123]:
# Run after cell 46
rem = cross_left.copy()
rem["band"] = np.select(
    [rem["token_jaccard"].ge(0.9), rem["token_jaccard"].ge(0.8)],
    ["jac>=0.90", "jac 0.80-0.90"], default="jac<0.80")
print(rem.groupby(["band", "labels_conflict"]).size().unstack(fill_value=0))
exposed = set(rem.loc[rem.split_a.eq("test"), "id_a"]) | set(rem.loc[rem.split_b.eq("test"), "id_b"])
print("test IDs still exposed:", len(exposed))

for band, g in rem.groupby("band"):
    print("\n#####", band, len(g))
    for n, (_, r) in enumerate(g.sample(n=min(25, len(g)), random_state=42).iterrows()):
        print(f"\n[{band} #{n}] sim={r.similarity:.3f} {r.split_a}/{r.label_a} vs {r.split_b}/{r.label_b}")
        review_pair(r.source_a, r.id_a, r.source_b, r.id_b)

Empty DataFrame
Columns: []
Index: []
test IDs still exposed: 0


## Interpretation checkpoints — do not train yet

## Save aggregate audit outputs

The reports directory contains aggregate counts, IDs and hashes only. No raw tweet text is copied to Drive.

In [124]:
# HinEmo aggregate outputs
hinemo_language_tier_counts.to_csv(REPORT_DIR / "hinemo_language_tier_counts.csv", index=False)
hinemo_language_label_counts.to_csv(REPORT_DIR / "hinemo_language_label_counts.csv", index=False)
hinglish_train_tier_label.to_csv(REPORT_DIR / "hinemo_hinglish_train_tier_label_counts.csv", index=False)
hinglish_eval_counts.to_csv(REPORT_DIR / "hinemo_hinglish_eval_label_counts.csv", index=False)
hinemo_hash_audit.to_csv(REPORT_DIR / "hinemo_checksum_audit.csv", index=False)
hinemo_duplicate_audit.to_csv(REPORT_DIR / "hinemo_duplicate_audit_ids_only.csv", index=False)
(REPORT_DIR / "hinemo_integrity_summary.json").write_text(
    json.dumps(hinemo_integrity, indent=2, ensure_ascii=False), encoding="utf-8"
)

# EmoSen aggregate outputs (IDs/counts/hashes only; no raw tweet text)
emosen_emotion_counts.to_csv(REPORT_DIR / "emosen_emotion_counts.csv")
emosen_sentiment_counts.to_csv(REPORT_DIR / "emosen_sentiment_counts.csv")
emosen_schema_audit.to_csv(REPORT_DIR / "emosen_schema_audit.csv", index=False)
paper_csv_comparison.to_csv(REPORT_DIR / "emosen_paper_vs_csv_validation_counts.csv", index=False)
(REPORT_DIR / "emosen_integrity_summary.json").write_text(
    json.dumps(emosen_integrity, indent=2, ensure_ascii=False), encoding="utf-8"
)

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}
(REPORT_DIR / "hinglish_audit_environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

print("Audit reports saved to:", REPORT_DIR)
print(json.dumps(environment, indent=2))


Audit reports saved to: /content/reports
{
  "python": "3.13.18",
  "pandas": "3.1.1",
  "numpy": "2.4.6"
}


## 9. Interpretation checkpoints — do not train yet

Before Phase 2 data is frozen, resolve these items:

1. **HinEmo-9 `full.jsonl`:** in the uploaded archive, all rows are marked `llm`; for the 3,363 gold rows its `label` matches the split file's `llm_prediction`, not necessarily the final gold label. There are 772 label differences between `full.jsonl` and the split files. Use `train.jsonl`, `dev.jsonl`, and `test.jsonl` for the pipeline; treat `full.jsonl` as a pre-final prediction snapshot unless maintainers clarify otherwise.
2. **HinEmo-9 checksum line endings:** the supplied hashes may correspond to CRLF rather than LF bytes. The checksum cell checks both forms.
3. **HinEmo-9 label coverage:** Romanized Hinglish has no `Disgust` examples. Current common-label evaluation can cover Anger, Fear, Joy, Sadness, Surprise and Neutral only. Love, Nostalgia and Devotion remain unmapped.
4. **HinEmo-9 test support:** inspect the gold Romanized Hinglish test support before interpreting macro-F1; several classes are small.
5. **EmoSen validation counts:** uploaded CSV counts differ from the paper's printed validation tables. Keep both records and seek clarification before selecting the validation split for model selection.
6. **EmoSen duplicates:** a normalized duplicate with conflicting emotion labels should be investigated; do not silently retain both for training.
7. **EmoSen access:** confirm that your use is covered by the compliance terms. Do not redistribute the data.
8. **Custom test leakage:** once the independently annotated custom test set exists, compare normalized text against every training/adaptation source before training.

Only after these checks should we freeze Hinglish training data and proceed to model implementation.


10. **Encoding review:** two EmoSen rows remain flagged by the diagnostic (`11831`, `11511`) and are intentionally left unchanged. This is a documented uncertainty, not a claim that the text is repaired.
11. **Similarity review:** the old 60-pair sample and previous drop counts are stale after the encoding freeze. Review the newly generated sample before enabling similarity buckets.
12. **Cluster chaining:** components larger than four members are excluded from automatic drop proposals and must be reviewed manually.
13. **Cloud storage:** no raw text is copied to Google Drive by this notebook. Store processed text only in a location explicitly permitted by the dataset terms.


In [125]:
from google.colab import files
import shutil
shutil.make_archive("/content/emosen_processed", "zip", "/content/data/processed")
files.download("/content/emosen_processed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [128]:
import os
import shutil

source = "/content/reports"
destination = "/content/drive/MyDrive/reports_02_hinglish_dataset_audit_with_encoding_diagnostic"

# Mount Google Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(destination, exist_ok=True)

copied = []
already_exists = []

for root, dirs, files in os.walk(source):
    for filename in files:
        src_path = os.path.join(root, filename)

        relative_path = os.path.relpath(src_path, source)
        dst_path = os.path.join(destination, relative_path)

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)
            copied.append(relative_path)
        else:
            already_exists.append(relative_path)

print("New files copied:", len(copied))
print("Files already present:", len(already_exists))

print("\nNewly copied files:")
for f in copied:
    print(f)

print("\nSUCCESS: Missing files copied to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
New files copied: 1
Files already present: 22

Newly copied files:
emosen_processed.zip

SUCCESS: Missing files copied to Google Drive.
